# **10 ВАРИАНТ - InceptionV3** Танюшкин А.Л.

# Лабораторная работа №3. Классификация изображений CNN

	Цель лабораторной работы – Классификация изображений с использованием свёрточных нейронных сетей.

1.Определить какую сеть вам нужно будет загрузить согласно вашему номеру варианта:

# InceptionV3

2.Загрузить сеть, предобученную на ImageNet (https://keras.io/api/applications/)

In [2]:
import os
import numpy as np
import pandas as pd

import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.applications import InceptionV3 # type: ignore
from tensorflow.keras.applications.inception_v3 import preprocess_input, decode_predictions # type: ignore
from tensorflow.keras.preprocessing import image # type: ignore

from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense # type: ignore

In [3]:
model = InceptionV3(include_top = True,
                    weights='imagenet',
                    input_shape = (299, 299, 3),
                    classes = 1000
                    )

3.Выполнить классификацию изображения на загруженной сети, найденного в интернете. Изображение должно показывать один объект одного из следующих классов: лошадь, собака, кошка, самолёт, корабль, автомобиль, дом

In [4]:
def classify_img():
    for file in os.listdir('task2/'):
        file_p = os.path.join('task2/', file)

        img = image.load_img(file_p, target_size=(299, 299))

        img_arr = image.img_to_array(img)
        img_arr = np.expand_dims(img_arr, axis=0)
        img_arr = preprocess_input(img_arr)

        pred = model.predict(img_arr)
        decod_pred = decode_predictions(pred, top=3)[0]

        for i, prediction in enumerate(decod_pred):
            print(f"{i+1}. {prediction[1]}: {prediction[2]:.2%}            Файл: {file}")
        

In [5]:
classify_img()

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1. minivan: 33.12%            Файл: images.jpeg
2. sports_car: 4.97%            Файл: images.jpeg
3. pole: 4.28%            Файл: images.jpeg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
1. washer: 96.44%            Файл: images12.jpeg
2. ashcan: 0.15%            Файл: images12.jpeg
3. stove: 0.13%            Файл: images12.jpeg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
1. moving_van: 10.55%            Файл: images2.jpeg
2. minibus: 4.73%            Файл: images2.jpeg
3. ambulance: 3.53%            Файл: images2.jpeg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
1. bullet_train: 26.99%            Файл: images3.jpeg
2. passenger_car: 24.86%            Файл: images3.jpeg
3. streetcar: 15.77%            Файл: images3.jpeg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
1. minivan: 13.17%            Файл: images4.jpeg
2. moving_van: 5.04%            Файл: images4.jpeg
3. limousine: 4.59%            Файл: images4.jpeg
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
1. cab: 20.78%            

4.	Загрузить два набора данных horses_or_humans, cats_vs_dogs

In [6]:
(train_cat_dog, test_cat_dog) = tfds.load('cats_vs_dogs',
                                split=['train[:70%]', 'train[70%:]'],
                                shuffle_files=True, 
                                as_supervised=True)
train_cat_dog = train_cat_dog.take(1227)
test_cat_dog = test_cat_dog.take(256)

In [7]:
(train_horse_human, test_horse_human) = tfds.load('horses_or_humans',
                                split=['train', 'test'],
                                shuffle_files=True,
                                as_supervised=True)

In [8]:
def preprocess_for_inc(image, label):
    image = tf.image.resize(image, [224, 224])
    image = tf.cast(image, tf.float32) / 255.0

    return image, label

In [9]:
train_horse_human = train_horse_human.map(
        lambda x, y: preprocess_for_inc(x, y)
    )
test_horse_human = test_horse_human.map(
        lambda x, y: preprocess_for_inc(x, y)
    )

train_cat_dog = train_cat_dog.map(
        lambda x, y: preprocess_for_inc(x, y)
    )
test_cat_dog = test_cat_dog.map(
        lambda x, y: preprocess_for_inc(x, y)
    )

In [10]:
val_horse_human = train_horse_human.take(200)
val_cat_dog = train_cat_dog.take(200)

train_horse_human = train_horse_human.skip(200)
train_cat_dog = train_cat_dog.skip(200)

In [11]:
train_horse_human = train_horse_human.batch(32).prefetch(tf.data.AUTOTUNE)
test_horse_human = test_horse_human.batch(32).prefetch(tf.data.AUTOTUNE)
val_horse_human = val_horse_human.batch(32).prefetch(tf.data.AUTOTUNE)

train_cat_dog = train_cat_dog.batch(32).prefetch(tf.data.AUTOTUNE)
test_cat_dog = test_cat_dog.batch(32).prefetch(tf.data.AUTOTUNE)
val_cat_dog = val_cat_dog.batch(32).prefetch(tf.data.AUTOTUNE)

In [12]:
train_cat_dog

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>

In [13]:
train_horse_human

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>

7.Создать собственную сеть (полностью своя сеть со своими слоями и архитектурой), обучить данную сеть с помощью генераторов на двух наборах данных: horses_or_humans, cats_vs_dogs. Оценить точность классификации сети. (за качеством не гонимся) 

In [14]:
model = Sequential()

model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)))
model.add(MaxPooling2D(2, 2))
model.add(Dropout(0.25))

model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))
model.add(Dropout(0.25))

model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))
model.add(Dropout(0.25))

model.add(Conv2D(256, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))
model.add(Dropout(0.25))

model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

c:\Users\Aleggg\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [15]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
    )

In [16]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_94 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_95 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_96 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_97 (Conv2D)              │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │    18,874,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,263,809 (73.49 MB)

 Trainable params: 19,263,809 (73.49 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
history_horses = model.fit(
    train_horse_human,
    epochs=10,
    validation_data=val_horse_human,
    verbose=1
)

Epoch 1/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 28s 998ms/step - accuracy: 0.5272 - loss: 1.2633 - val_accuracy: 0.5750 - val_loss: 0.6929
Epoch 2/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 30s 1s/step - accuracy: 0.6155 - loss: 0.6587 - val_accuracy: 0.7600 - val_loss: 0.6157
Epoch 3/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 27s 1s/step - accuracy: 0.7074 - loss: 0.5727 - val_accuracy: 0.7700 - val_loss: 0.5852
Epoch 4/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 27s 1s/step - accuracy: 0.8041 - loss: 0.4351 - val_accuracy: 0.8450 - val_loss: 0.3854
Epoch 5/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 24s 903ms/step - accuracy: 0.8682 - loss: 0.3034 - val_accuracy: 0.8400 - val_loss: 0.3211
Epoch 6/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 23s 885ms/step - accuracy: 0.9166 - loss: 0.1968 - val_accuracy: 0.9850 - val_loss: 0.1110
Epoch 7/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 23s 892ms/step - accuracy: 0.9649 - loss: 0.0916 - val_accuracy: 0.9850 - val_loss: 0.0763
Epoch 8/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 23s 893ms/step - accuracy: 0.9698 - loss: 0.0663 - val_accuracy: 1.0

In [18]:
test_loss_horses, test_acc_horses = model.evaluate(test_horse_human)
print(f'точность horses_or_humans: {test_acc_horses:.2%}')

8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 175ms/step - accuracy: 0.8242 - loss: 0.8396
точность horses_or_humans: 82.42%


In [19]:
model2 = Sequential()

model2.add(Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)))
model2.add(MaxPooling2D(2, 2))
model2.add(Dropout(0.25))

model2.add(Conv2D(64, (3, 3), activation='relu'))
model2.add(MaxPooling2D(2, 2))
model2.add(Dropout(0.25))

model2.add(Conv2D(128, (3, 3), activation='relu'))
model2.add(MaxPooling2D(2, 2))
model2.add(Dropout(0.25))

model2.add(Conv2D(256, (3, 3), activation='relu'))
model2.add(MaxPooling2D(2, 2))
model2.add(Dropout(0.25))

model2.add(Flatten())
model2.add(Dense(512, activation='relu'))
model2.add(Dropout(0.5))
model2.add(Dense(1, activation='sigmoid'))

In [20]:
model2.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
    )

In [21]:
model2.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_98 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_99 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_100 (Conv2D)             │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_101 (Conv2D)             │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │    18,874,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,263,809 (73.49 MB)

 Trainable params: 19,263,809 (73.49 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
history_cat = model2.fit(
    train_cat_dog,
    epochs=5,
    validation_data=val_cat_dog,
    verbose=1
)

Epoch 1/5
33/33 ━━━━━━━━━━━━━━━━━━━━ 29s 832ms/step - accuracy: 0.4937 - loss: 0.9584 - val_accuracy: 0.4500 - val_loss: 0.6932
Epoch 2/5
33/33 ━━━━━━━━━━━━━━━━━━━━ 27s 822ms/step - accuracy: 0.5054 - loss: 0.6932 - val_accuracy: 0.4500 - val_loss: 0.6938
Epoch 3/5
33/33 ━━━━━━━━━━━━━━━━━━━━ 27s 823ms/step - accuracy: 0.5180 - loss: 0.6930 - val_accuracy: 0.4550 - val_loss: 0.6940
Epoch 4/5
33/33 ━━━━━━━━━━━━━━━━━━━━ 28s 831ms/step - accuracy: 0.5258 - loss: 0.6929 - val_accuracy: 0.4650 - val_loss: 0.6941
Epoch 5/5
33/33 ━━━━━━━━━━━━━━━━━━━━ 27s 818ms/step - accuracy: 0.5268 - loss: 0.6925 - val_accuracy: 0.4500 - val_loss: 0.6955


In [23]:
test_loss_cats, test_acc_cats = model2.evaluate(test_cat_dog)
print(f'точность cats_vs_dogs: {test_acc_cats:.2%}')

8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 182ms/step - accuracy: 0.5039 - loss: 0.6932
точность cats_vs_dogs: 50.39%


8.Загрузить сеть, предобученную на InceptionV3 по варианту из пункта 1, выполнить дообучение этой сети на двух наборах данных: horses_or_humans, cats_vs_dogs. Оценить точность классификации новой (составной) сети. (А вот здесь с качеством можно поработать)

9.Оценка классификации должна соответствовать следующим требованиям:

a.	Должна быть confusion matrix

b.	Должны быть показаны метрики accuracy, recall, precision, f1

c.	Должна быть построена PR кривая для каждого класса

# Доработанный вариант в файле "lab3(3_0).ipynb"